<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#This is code to manage dependencies if the notebook is executed in the google colab cloud service
if 'google.colab' in str(get_ipython()):
  import os
  os.system('apt -qqq install graphviz')
  os.system('pip -qqq install ModelFlowIb')

In [2]:
import pandas as pd
import re

from modelclass import model
import modelmanipulation as mp
from modelmanipulation import doable, explode

from modelpattern import list_extract
import modeljupytermagic

In [3]:
from modelhelp import colab_link
colab_link('Specification_debt_simulation_model',badge=True,render=0)

<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# About lists

**lists** and **sublists** are central to making ModelFlow scalable and expressive.  
They allow the modeler to define **groups of entities** (such as bonds, banks, portfolios, sectors, or countries) and to attach **attributes** or **hierarchies** to those entities.  
A list definition typically takes the following form

$list\: \underbrace{issued}_{List name} = \underbrace{issued}_{sub listname} : issued\_2025 * issued\_2050 / \\
\underbrace{issued\_year}_{sub listname} : 2025 * 2050$

Please note:
 - the list name and the first sub list name should be the same
 - case don't matter
 - a * combined with identifiers ending in digits will construct multiple list content

  `issued_2025 * issued_2050` becomes `issued_2025 issued_2026 … issued_2050`


This enables equations to be written once as templates and then **expanded automatically** across all relevant list members.  

Sublists can be used to define logical relationships between entities — for example, linking each bond to its home maturity, or to define the year for each issurance year.  
These structures make it possible to organize models systematically and to maintain consistency across entities that share the same behavioral equations.  

This approach eliminates redundancy and manual repetition: instead of writing dozens or hundreds of similar equations, the modeler defines a **single equation pattern**, and ModelFlow expands it dynamically over the members of a list.  

In [4]:
%%Mexplodemodel port segment=listbonds
# List definitions


> list issued = issued        : issued_2025 * issued_2050 /
>>              issued_year   :        2025 *        2050

>list bond = bond :     1_year_dom 5_year_dom 10_year_dom  1_year_usd 5_year_usd 10_year_usd     /
>>        maturity :             1           5          10           1          5          10    /
>>        grace    :              0          3          6           0         4            9     /
>>        currency :            dom        dom         dom         usd        usd         usd   /
>>         domestic :             1          1           1           0          0           0   /
>>         foreign :              0          0           0           1          1           1  

>list currency = currency : dom usd    

# List definitions


```text
> list issued = issued        : issued_2025 * issued_2050 /
>>              issued_year   :        2025 *        2050
```

```text
>list bond = bond :     1_year_dom 5_year_dom 10_year_dom  1_year_usd 5_year_usd 10_year_usd     /
>>        maturity :             1           5          10           1          5          10    /
>>        grace    :              0          3          6           0         4            9     /
>>        currency :            dom        dom         dom         usd        usd         usd   /
>>         domestic :             1          1           1           0          0           0   /
>>         foreign :              0          0           0           1          1           1  
```

```text
>list currency = currency : dom usd    
```

In [6]:
port_replacements=[('__bond','__{bond}__{issued}'),
                   ('fx_rate','dom_{currency}')
                  ]

In [7]:
%%Mexplodemodel port segment=logical replacements=port_replacements nshow render=0 render_list=False

## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

> doable logical_issuing__bond = current_year == {issued_year}
> doable logical_maturing__bond =  current_year == {issued_year}+{maturity}
> doable logical_amortizing__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})
> doable logical_paying_years__bond =  {issued_year} < current_year <= ({issued_year} + {maturity})
> doable amortizing_time__bond =  {maturity}-{grace}
> doable amortizing_share_pr_time__bond =  logical_amortizing__bond * 1./amortizing_time__bond

In [8]:
%%Mexplodemodel port segment=funding  replacements=port_replacements  nshow render=0 nrender_list=False 

## Funding need
>funding_need = deficit+amortizing__all + interest_payments__all

In [9]:
%%Mexplodemodel port segment=issurance  replacements=port_replacements  nshow render=0 nrender_list=False 


### Domestic and foreign debt 
> bond_share_foreign = 100 - bond_share_dom
> issued_domestic = bond_share_dom/100 * funding_need
> issued_foreign  = bond_share_foreign/100 * funding_need

### issurance in domestic currency 
> doable [bond=domestic] new_issue__bond  = new_issue_share__{bond}/100 * issued_domestic * logical_issuing__bond 
> doable  [bond=foreign] new_issue__bond = new_issue_share__{bond}/100 * issued_foreign  * logical_issuing__bond  

### issurance as percent of total now 
> doable share_of_funding__bond = new_issue__bond /funding_need 

### issurance in fx  currency. dom_dom = 1 

> doable new_issue_in_currency__bond = new_issue__bond / fx_rate 

### original issurance in fx  currency to calculate amortization  

> doable  original_outstanding_in_currency__bond = (new_issue_in_currency__bond *logical_issuing__bond+
>>            original_outstanding_in_currency__bond(-1)*logical_paying_years__bond) 



In [10]:
%%Mexplodemodel port segment=stock replacements=port_replacements  nshow render=0 render_list=False

### Amortizing in fx 
> doable  amortizing_in_currency__bond = original_outstanding_in_currency__bond * amortizing_share_pr_time__bond

### End of year stock in fx
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond+ new_issue_in_currency__bond-amortizing_in_currency__bond

### Start of year stock in fx 
> doable stock_primo_in_currency__bond = stock_ultimo_in_currency__bond(-1)

### Amortizing in domestic currency 

> doable <sum=all>  amortizing__bond = amortizing_in_currency__bond * fx_rate

### Stock in domestic currency 
> doable <sum=all> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=all> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate

In [11]:
%%Mexplodemodel port segment=interest replacements=port_replacements  nshow render=0 render_list=False

## Interest rates
The first year after the issurance the interest rate is taken from the interest rate of the relevant maturity.

In the rest of the paying years the interest rate are set to the bonds interest rate the previous year
> doable  interest_rate__bond = interest_rate__{currency}__{maturity}(-1) * logical_issuing__bond (-1) + interest_rate__bond(-1) * logical_paying_years__bond


## Foreign bonds 


### Interest rate payment
> doable interest_in_currency__bond = stock_primo_in_currency__bond * interest_rate__bond/100

## Domestic
### Interest rate payment
> doable <sum=all>  interest_payments__bond    = interest_in_currency__bond  * fx_rate


In [12]:
%Mexplodemodel port replacements=port_replacements
mport = port.mmodel


## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

```text
> doable logical_issuing__bond = current_year == {issued_year}
> doable logical_maturing__bond =  current_year == {issued_year}+{maturity}
> doable logical_amortizing__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})
> doable logical_paying_years__bond =  {issued_year} < current_year <= ({issued_year} + {maturity})
> doable amortizing_time__bond =  {maturity}-{grace}
> doable amortizing_share_pr_time__bond =  logical_amortizing__bond * 1./amortizing_time__bond
```


## Funding need
```text
>funding_need = deficit+amortizing__all + interest_payments__all
```



### Domestic and foreign debt 
```text
> bond_share_foreign = 100 - bond_share_dom
> issued_domestic = bond_share_dom/100 * funding_need
> issued_foreign  = bond_share_foreign/100 * funding_need
```

### issurance in domestic currency 
```text
> doable [bond=domestic] new_issue__bond  = new_issue_share__{bond}/100 * issued_domestic * logical_issuing__bond 
> doable  [bond=foreign] new_issue__bond = new_issue_share__{bond}/100 * issued_foreign  * logical_issuing__bond  
```

### issurance as percent of total now 
```text
> doable share_of_funding__bond = new_issue__bond /funding_need 
```

### issurance in fx  currency. dom_dom = 1 

```text
> doable new_issue_in_currency__bond = new_issue__bond / fx_rate 
```

### original issurance in fx  currency to calculate amortization  

```text
> doable  original_outstanding_in_currency__bond = (new_issue_in_currency__bond *logical_issuing__bond+
>>            original_outstanding_in_currency__bond(-1)*logical_paying_years__bond) 
```



### Amortizing in fx 
```text
> doable  amortizing_in_currency__bond = original_outstanding_in_currency__bond * amortizing_share_pr_time__bond
```

### End of year stock in fx
```text
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond+ new_issue_in_currency__bond-amortizing_in_currency__bond
```

### Start of year stock in fx 
```text
> doable stock_primo_in_currency__bond = stock_ultimo_in_currency__bond(-1)
```

### Amortizing in domestic currency 

```text
> doable <sum=all>  amortizing__bond = amortizing_in_currency__bond * fx_rate
```

### Stock in domestic currency 
```text
> doable <sum=all> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=all> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate
```


## Interest rates
The first year after the issurance the interest rate is taken from the interest rate of the relevant maturity.

In the rest of the paying years the interest rate are set to the bonds interest rate the previous year
```text
> doable  interest_rate__bond = interest_rate__{currency}__{maturity}(-1) * logical_issuing__bond (-1) + interest_rate__bond(-1) * logical_paying_years__bond
```


## Foreign bonds 


### Interest rate payment
```text
> doable interest_in_currency__bond = stock_primo_in_currency__bond * interest_rate__bond/100
```

## Domestic
### Interest rate payment
```text
> doable <sum=all>  interest_payments__bond    = interest_in_currency__bond  * fx_rate
```

# List definitions


```text
> list issued = issued        : issued_2025 * issued_2050 /
>>              issued_year   :        2025 *        2050
```

```text
>list bond = bond :     1_year_dom 5_year_dom 10_year_dom  1_year_usd 5_year_usd 10_year_usd     /
>>        maturity :             1           5          10           1          5          10    /
>>        grace    :              0          3          6           0         4            9     /
>>        currency :            dom        dom         dom         usd        usd         usd   /
>>         domestic :             1          1           1           0          0           0   /
>>         foreign :              0          0           0           1          1           1  
```

```text
>list currency = currency : dom usd    
```

In [13]:
# Segments
port_dict.keys() 

dict_keys(['listbonds', 'logical', 'funding', 'issurance', 'stock', 'interest'])

In [14]:
mport.exogene

{'BOND_SHARE_DOM',
 'CURRENT_YEAR',
 'DEFICIT',
 'DOM_DOM',
 'DOM_USD',
 'INTEREST_RATE__DOM__1',
 'INTEREST_RATE__DOM__10',
 'INTEREST_RATE__DOM__5',
 'INTEREST_RATE__USD__1',
 'INTEREST_RATE__USD__10',
 'INTEREST_RATE__USD__5',
 'NEW_ISSUE_SHARE__10_YEAR_DOM',
 'NEW_ISSUE_SHARE__10_YEAR_USD',
 'NEW_ISSUE_SHARE__1_YEAR_DOM',
 'NEW_ISSUE_SHARE__1_YEAR_USD',
 'NEW_ISSUE_SHARE__5_YEAR_DOM',
 'NEW_ISSUE_SHARE__5_YEAR_USD'}

In [15]:
years = [y for y in range(2025,2050,1)]
dfstart = pd.DataFrame(years,index=years,columns=['CURRENT_YEAR'])
dfstart

,CURRENT_YEAR
2025,2025
2026,2026
2027,2027
2028,2028
2029,2029
2030,2030
2031,2031
2032,2032
2033,2033
2034,2034


In [16]:
df = dfstart.upd('''
bond_share_dom = 100 
new_issue_share__1_year_dom = 0
new_issue_share__5_year_dom  = 100
new_issue_share__10_year_dom  = 0
interest_rate__dom__1 = 1
interest_rate__dom__5 = 1
interest_rate__dom__10 = 1
interest_rate_usd__1 = 3
interest_rate_usd__5 = 3
interest_rate_usd__10 = 3
dom_dom = 1
dom_usd = 6
<2026> deficit  = 100

''')
#df


In [17]:
res = mport(df,2026,2050,silent=0)

Will start solving: testmodel
New data or transpile_reset
Create compiled solving function for testmodel
ljit=False stringjit=False  transpile_reset=False  hasattr(self, f"pro_{jitname}")=False
now makelos makes a sim solvefunction
2026 Solved in 6 iterations
2027 Solved in 6 iterations
2028 Solved in 6 iterations
2029 Solved in 6 iterations
2030 Solved in 6 iterations
2031 Solved in 6 iterations
2032 Solved in 6 iterations
2033 Solved in 6 iterations
2034 Solved in 6 iterations
2035 Solved in 6 iterations
2036 Solved in 6 iterations
2037 Solved in 6 iterations
2038 Solved in 6 iterations
2039 Solved in 6 iterations
2040 Solved in 6 iterations
2041 Solved in 6 iterations
2042 Solved in 6 iterations
2043 Solved in 6 iterations
2044 Solved in 6 iterations
2045 Solved in 6 iterations
2046 Solved in 6 iterations
2047 Solved in 6 iterations
2048 Solved in 6 iterations
2049 Solved in 6 iterations
testmodel solved  


In [18]:
mport['funding*  interest_in_currency__*26'].df.T

,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
FUNDING_NEED,100.0,1.0,1.01,1.0201,51.030301,51.540604,2.05601,2.07657,27.097336,52.368309,...,41.268244,16.680926,10.597736,29.453713,42.24825,30.170733,14.84744,21.245914,37.083374,37.454207
INTEREST_IN_CURRENCY__10_YEAR_DOM__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__10_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__1_YEAR_DOM__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__1_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__5_YEAR_DOM__ISSUED_2026,0.0,1.0,1.00,1.0000,1.000000,0.500000,0.00000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__5_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000


In [19]:
mport['amortizing*5*DOM*26 ORIGINAL_ISSUE_IN_CURRENCY__5_YEAR_DOM__ISSUED_2026 interest_rate*26'].df.T


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
AMORTIZING_IN_CURRENCY__5_YEAR_DOM__ISSUED_2026,0.0,0.0,0.0,0.0,50.0,50.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING_SHARE_PR_TIME__5_YEAR_DOM__ISSUED_2026,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING_TIME__5_YEAR_DOM__ISSUED_2026,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
AMORTIZING__5_YEAR_DOM__ISSUED_2026,0.0,0.0,0.0,0.0,50.0,50.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
INTEREST_RATE__10_YEAR_DOM__ISSUED_2026,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
INTEREST_RATE__10_YEAR_USD__ISSUED_2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
INTEREST_RATE__1_YEAR_DOM__ISSUED_2026,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
INTEREST_RATE__1_YEAR_USD__ISSUED_2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
INTEREST_RATE__5_YEAR_DOM__ISSUED_2026,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
INTEREST_RATE__5_YEAR_USD__ISSUED_2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
mport.AMORTIZING_SHARE_PR_TIME__5_YEAR_DOM__ISSUED_2026	

Endogeneous: AMORTIZING_SHARE_PR_TIME__5_YEAR_DOM__ISSUED_2026: 
Formular: FRML <> AMORTIZING_SHARE_PR_TIME__5_YEAR_DOM__ISSUED_2026 = LOGICAL_AMORTIZING__5_YEAR_DOM__ISSUED_2026*1./AMORTIZING_TIME__5_YEAR_DOM__ISSUED_2026 $

AMORTIZING_SHARE_PR_TIME__5_YEAR_DOM__ISSUED_2026: 
AMORTIZING_TIME__5_YEAR_DOM__ISSUED_2026         : 
LOGICAL_AMORTIZING__5_YEAR_DOM__ISSUED_2026      : 

Values :


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
Base,0.00,0.00,0.00,0.00,0.50,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Last,0.00,0.00,0.00,0.00,0.50,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Diff,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Input last run:


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
AMORTIZING_TIME__5_YEAR_DOM__ISSUED_2026,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00
LOGICAL_AMORTIZING__5_YEAR_DOM__ISSUED_2026,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [21]:
mport. interest_rate__5_YEAR_DOM__ISSUED_2026

Endogeneous: INTEREST_RATE__5_YEAR_DOM__ISSUED_2026: 
Formular: FRML <> INTEREST_RATE__5_YEAR_DOM__ISSUED_2026 = INTEREST_RATE__DOM__5(-1)*LOGICAL_ISSUING__5_YEAR_DOM__ISSUED_2026(-1)+INTEREST_RATE__5_YEAR_DOM__ISSUED_2026(-1)*LOGICAL_PAYING_YEARS__5_YEAR_DOM__ISSUED_2026 $

INTEREST_RATE__5_YEAR_DOM__ISSUED_2026       : 
INTEREST_RATE__DOM__5                        : 
LOGICAL_ISSUING__5_YEAR_DOM__ISSUED_2026     : 
LOGICAL_PAYING_YEARS__5_YEAR_DOM__ISSUED_2026: 

Values :


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
Base,0.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Last,0.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Diff,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Input last run:


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
INTEREST_RATE__5_YEAR_DOM__ISSUED_2026(-1),0.00,0.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
INTEREST_RATE__DOM__5(-1),1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
LOGICAL_ISSUING__5_YEAR_DOM__ISSUED_2026(-1),0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
LOGICAL_PAYING_YEARS__5_YEAR_DOM__ISSUED_2026,0.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [22]:
mport['stock_primo__all stock_ultimo__all interest*__*26 funding* paying_years__*26'].df.T


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
STOCK_PRIMO__ALL,0.0,100.0,101.00,102.0100,103.030100,104.060401,105.101005,106.152015,107.213535,108.285671,...,113.809328,114.947421,116.096896,117.257864,118.430443,119.614748,120.810895,122.019004,123.239194,124.471586
STOCK_ULTIMO__ALL,100.0,101.0,102.01,103.0301,104.060401,105.101005,106.152015,107.213535,108.285671,109.368527,...,114.947421,116.096896,117.257864,118.430443,119.614748,120.810895,122.019004,123.239194,124.471586,125.716302
INTEREST_IN_CURRENCY__10_YEAR_DOM__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__10_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__1_YEAR_DOM__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__1_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__5_YEAR_DOM__ISSUED_2026,0.0,1.0,1.00,1.0000,1.000000,0.500000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_IN_CURRENCY__5_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_PAYMENTS__10_YEAR_DOM__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
INTEREST_PAYMENTS__10_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [23]:
mport['interest_payments__all interest_in_currency__*202? interest_rate__*202?'].df.head().T

,2026,2027,2028,2029,2030
INTEREST_PAYMENTS__ALL,0.0,1.0,1.01,1.0201,1.030301
INTEREST_IN_CURRENCY__10_YEAR_DOM__ISSUED_2025,0.0,0.0,0.00,0.0000,0.000000
INTEREST_IN_CURRENCY__10_YEAR_DOM__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000
INTEREST_IN_CURRENCY__10_YEAR_DOM__ISSUED_2027,0.0,0.0,0.00,0.0000,0.000000
INTEREST_IN_CURRENCY__10_YEAR_DOM__ISSUED_2028,0.0,0.0,0.00,0.0000,0.000000
...,...,...,...,...,...
INTEREST_RATE__5_YEAR_USD__ISSUED_2025,0.0,0.0,0.00,0.0000,0.000000
INTEREST_RATE__5_YEAR_USD__ISSUED_2026,0.0,0.0,0.00,0.0000,0.000000
INTEREST_RATE__5_YEAR_USD__ISSUED_2027,0.0,0.0,0.00,0.0000,0.000000
INTEREST_RATE__5_YEAR_USD__ISSUED_2028,0.0,0.0,0.00,0.0000,0.000000


In [24]:
mport.NEW_ISSUE_IN_CURRENCY__1_YEAR_DOM__ISSUED_2026

Endogeneous: NEW_ISSUE_IN_CURRENCY__1_YEAR_DOM__ISSUED_2026: 
Formular: FRML <> NEW_ISSUE_IN_CURRENCY__1_YEAR_DOM__ISSUED_2026 = NEW_ISSUE__1_YEAR_DOM__ISSUED_2026/DOM_DOM $

NEW_ISSUE_IN_CURRENCY__1_YEAR_DOM__ISSUED_2026: 
DOM_DOM                                       : 
NEW_ISSUE__1_YEAR_DOM__ISSUED_2026            : 

Values :


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
Base,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Last,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Diff,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Input last run:


,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
DOM_DOM,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
NEW_ISSUE__1_YEAR_DOM__ISSUED_2026,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [25]:
mport.modeldump('model/port')

In [26]:
mport['amort*__5*'].df.T

,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
AMORTIZING_IN_CURRENCY__5_YEAR_DOM__ISSUED_2025,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING_IN_CURRENCY__5_YEAR_DOM__ISSUED_2026,0.0,0.0,0.0,0.0,50.0,50.0,0.000,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING_IN_CURRENCY__5_YEAR_DOM__ISSUED_2027,0.0,0.0,0.0,0.0,0.0,0.5,0.500,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING_IN_CURRENCY__5_YEAR_DOM__ISSUED_2028,0.0,0.0,0.0,0.0,0.0,0.0,0.505,0.50500,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING_IN_CURRENCY__5_YEAR_DOM__ISSUED_2029,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.51005,0.51005,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AMORTIZING__5_YEAR_USD__ISSUED_2045,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING__5_YEAR_USD__ISSUED_2046,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING__5_YEAR_USD__ISSUED_2047,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMORTIZING__5_YEAR_USD__ISSUED_2048,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00000,0.00000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
